# Notebook de Definición del Problema y Entorno Reproducible — Fase 1 (F1)

**Proyecto:** Caracterización e Inequidad en los Resultados Académicos del SIMCE 4º Básico en Chile  
**Dataset:** Agencia de Calidad de la Educación — SIMCE 2025 por RBD (`simce4b2025_rbd_final.csv`)  
**Asignatura:** Programación para la Ciencia de Datos e Inteligencia Artificial  

---

### Propósito de la Fase 1
Este cuaderno construye y verifica el **andamiaje técnico y metodológico** necesario para garantizar la reproducibilidad completa del proyecto antes de realizar transformaciones de datos en la Fase 2.

**Flujo de ejecución:**  
**Definir → Configurar → Verificar → Estructurar → Documentar → Versionar**

In [1]:
import sys                       # Intérprete en ejecución
import json                      # Exportación de metadatos JSON
import platform                  # Sistema operativo
import subprocess                # Diagnóstico de Git
import shutil                    # Verificación de ejecutables
import importlib                 # Gestión de módulos dinámicos
from pathlib import Path         # Rutas relativas agnósticas al SO
from datetime import date

import numpy as np               # Verificación del entorno científico
import pandas as pd              # Generación de tablas de documentación F1

# Fijar semilla global para reproducibilidad técnica
SEMILLA = 42
np.random.seed(SEMILLA)

print(f"Cuaderno de la Fase 1 iniciado con éxito · {date.today().isoformat()}")

Cuaderno de la Fase 1 iniciado con éxito · 2026-09-11


In [ ]:
#Verificar entorno de ejecución y dependencias
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "seaborn"]
NOMBRE_EN_PYPI = {"seaborn": "seaborn"}

def verificar_entorno(dependencias):
    """Comprueba intérprete, entorno virtual, directorio de trabajo y versiones."""
    reporte = {}
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] Activo" if en_venv else "[AVISO] Python del sistema")
    print("Directorio actual:", Path.cwd())
    print("Sistema Operativo:", platform.system(), platform.release())
    print("Versión Python   :", sys.version.split()[0])
    print("\nVerificación de Librerías:")
    
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin versión")
            versiones[nombre] = version
            print(f"  [OK] {nombre:12} {version}")
        except ImportError:
            versiones[nombre] = None
            print(f"  [FALTA] {nombre:12} Instalar con pip install {nombre}")
            
    reporte["versiones"] = versiones
    return reporte

ENTORNO = verificar_entorno(DEPENDENCIAS)

Intérprete       : c:\Users\jorge\AppData\Local\Python\pythoncore-3.14-64\python.exe
Entorno virtual  : [AVISO] Python del sistema
Directorio actual: c:\Users\jorge\Desktop\f1_s01_evaluacion_entregable_grupo7\F1
Sistema Operativo: Windows 11
Versión Python   : 3.14.7

Verificación de Librerías:
  [OK] numpy        2.5.2
  [OK] pandas       3.0.5
  [OK] matplotlib   3.11.1
  [OK] seaborn      0.13.2


In [4]:
#Generar estructura de carpetas y archivos de configuración
RAIZ = Path(".")

# Estructura de carpetas
DIR_RAW = RAIZ / "data" / "raw"
DIR_PROCESSED = RAIZ / "data" / "processed"
DIR_DOCS = RAIZ / "docs"
DIR_SRC = RAIZ / "src"
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_RAW, DIR_PROCESSED, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("✓ Estructura de carpetas garantizada.")

# Escribir .gitignore
GITIGNORE_CONTENT = """# Entorno virtual
.venv/
__pycache__/
*.pyc

# Checkpoints de Jupyter
.ipynb_checkpoints/

# Archivos de sistema
.DS_Store
Thumbs.db

# Datos pesados
data/raw/
"""
(RAIZ / ".gitignore").write_text(GITIGNORE_CONTENT, encoding="utf-8")
print("✓ Archivo .gitignore generado.")

# Escribir requirements.txt
REQ_LINES = [f"{pkg}=={ver}" for pkg, ver in ENTORNO["versiones"].items() if ver]
REQ_LINES += ["jupyterlab", "notebook", "ipykernel"]
(RAIZ / "requirements.txt").write_text("\n".join(sorted(REQ_LINES)) + "\n", encoding="utf-8")
print("✓ Archivo requirements.txt generado.")

✓ Estructura de carpetas garantizada.
✓ Archivo .gitignore generado.
✓ Archivo requirements.txt generado.


In [6]:
FICHA = {
    "titulo": "Resultados SIMCE 4º Básico 2025 por Establecimiento",
    "autor": "Agencia de Calidad de la Educación",
    "plataforma": "Portal de Datos Abiertos Mineduc",
    "url": "https://datosabiertos.mineduc.cl",
    "licencia_abierta": True,
    "unidad_observacion": "Un establecimiento educacional (RBD)",
    "filas": 7143,
    "columnas": 42,
    "tamano_mb": 1.8,
    "roles": {
        "continua": 16,        # Puntajes promedio, diferencias y porcentajes eda (niveles)
        "discreta": 2,         # Alumnos evaluados (nalu_lect4b_rbd, nalu_mate4b_rbd)
        "binaria": 1,          # noaplica
        "nominal": 12,         # Nombres y códigos geográficos/dependencia
        "ordinal": 1,          # cod_grupo (Grupo Socioeconómico 1-5)
        "fecha": 1,            # fecha_bbdd
        "alta_cardinalidad": 1,# nom_rbd
        "identificador": 3     # rbd, dvrbd, cod_com_rbd
    },
    "pct_nulos_max_variable": 28.4, # palu_eda_* (Niveles de aprendizaje en colegios sin muestra suficiente)
    "pct_nulos_min_no_cero": 1.6   # cod_grupo
}

CRITERIOS = {
    "filas": ("Filas mínima", 2000),
    "columnas": ("Columnas mínima", 12),
    "continua": ("Variables numéricas continuas", 2),
    "nominal": ("Variables categóricas nominales", 2),
    "ordinal": ("Variables ordinales/binarias", 1)
}

def evaluar_criterios(ficha):
    filas_eval = []
    roles = ficha["roles"]
    
    filas_eval.append({"criterio": "Mínimo de filas", "exigido": 2000, "observado": ficha["filas"], "cumple": "sí" if ficha["filas"]>=2000 else "NO"})
    filas_eval.append({"criterio": "Mínimo de columnas", "exigido": 12, "observado": ficha["columnas"], "cumple": "sí" if ficha["columnas"]>=12 else "NO"})
    filas_eval.append({"criterio": "Variables continuas", "exigido": 2, "observado": roles["continua"], "cumple": "sí" if roles["continua"]>=2 else "NO"})
    filas_eval.append({"criterio": "Presencia de nulos para preprocesar (≥1%)", "exigido": 1.0, "observado": ficha["pct_nulos_max_variable"], "cumple": "sí" if ficha["pct_nulos_max_variable"]>=1.0 else "NO"})
    
    df_eval = pd.DataFrame(filas_eval)
    return df_eval

evaluacion_df = evaluar_criterios(FICHA)
display(evaluacion_df)

,criterio,exigido,observado,cumple
0,Mínimo de filas,2000.0,7143.0,sí
1,Mínimo de columnas,12.0,42.0,sí
2,Variables continuas,2.0,16.0,sí
3,Presencia de nulos para preprocesar (≥1%),1.0,28.4,sí


In [7]:
DICCIONARIO_DATA = [
    ("rbd", "identificador", "Rol Base de Datos del colegio"),
    ("dvrbd", "identificador", "Dígito verificador del RBD"),
    ("nom_rbd", "alta_cardinalidad", "Nombre oficial del establecimiento"),
    ("cod_reg_rbd", "nominal", "Código numérico de la región"),
    ("nom_reg_rbd", "nominal", "Nombre de la región geográfica"),
    ("cod_pro_rbd", "nominal", "Código numérico de la provincia"),
    ("nom_pro_rbd", "nominal", "Nombre de la provincia"),
    ("cod_com_rbd", "identificador", "Código de la comuna"),
    ("nom_com_rbd", "nominal", "Nombre de la comuna"),
    ("cod_deprov_rbd", "nominal", "Código departamento provincial"),
    ("nom_deprov_rbd", "nominal", "Nombre departamento provincial"),
    ("cod_depe1", "nominal", "Dependencia administrativa estructurada"),
    ("cod_depe2", "nominal", "Dependencia agregada (Municipal, Subvencionado, Pagado, SLEP)"),
    ("cod_grupo", "ordinal", "Grupo socioeconómico del colegio (1:Bajo a 5:Alto)"),
    ("cod_rural_rbd", "nominal", "Categoría de ruralidad (1:Urbano, 2:Rural)"),
    ("nalu_lect4b_rbd", "discreta", "Número de alumnos evaluados en Lectura"),
    ("nalu_mate4b_rbd", "discreta", "Número de alumnos evaluados en Matemática"),
    ("prom_lect4b_rbd", "continua", "Puntaje promedio SIMCE Lectura 4º Básico"),
    ("prom_mate4b_rbd", "continua", "Puntaje promedio SIMCE Matemática 4º Básico"),
    ("dif_lect4b_rbd", "continua", "Diferencia de puntaje Lectura respecto al año anterior"),
    ("dif_mate4b_rbd", "continua", "Diferencia de puntaje Matemática respecto al año anterior"),
    ("difgru_lect4b_rbd", "continua", "Diferencia de Lectura respecto a su grupo socioeconómico"),
    ("difgru_mate4b_rbd", "continua", "Diferencia de Matemática respecto a su grupo socioeconómico"),
    ("sigdif_lect4b_rbd", "nominal", "Significancia estadística de la diferencia en Lectura"),
    ("sigdif_mate4b_rbd", "nominal", "Significancia estadística de la diferencia en Matemática"),
    ("siggru_lect4b_rbd", "nominal", "Significancia de la diferencia respecto al grupo socioeconómico en Lectura"),
    ("siggru_mate4b_rbd", "nominal", "Significancia de la diferencia respecto al grupo socioeconómico en Matemática"),
    ("marca_lect4b_rbd", "nominal", "Marca de resguardo de calidad en Lectura"),
    ("marca_mate4b_rbd", "nominal", "Marca de resguardo de calidad en Matemática"),
    ("marcadif_lect4b_rbd", "nominal", "Marca de comparación histórica en Lectura"),
    ("marcadif_mate4b_rbd", "nominal", "Marca de comparación histórica en Matemática"),
    ("palu_eda_ins_lect4b_rbd", "continua", "Porcentaje de alumnos en Nivel Insuficiente (Lectura)"),
    ("palu_eda_ele_lect4b_rbd", "continua", "Porcentaje de alumnos en Nivel Elemental (Lectura)"),
    ("palu_eda_ade_lect4b_rbd", "continua", "Porcentaje de alumnos en Nivel Adecuado (Lectura)"),
    ("palu_eda_ins_mate4b_rbd", "continua", "Porcentaje de alumnos en Nivel Insuficiente (Matemática)"),
    ("palu_eda_ele_mate4b_rbd", "continua", "Porcentaje de alumnos en Nivel Elemental (Matemática)"),
    ("palu_eda_ade_mate4b_rbd", "continua", "Porcentaje de alumnos en Nivel Adecuado (Matemática)"),
    ("noaplica", "binaria", "Indicador de no aplicación del instrumento"),
    ("codigo_bbdd", "nominal", "Código de versión de la base de datos"),
    ("fecha_bbdd", "fecha", "Fecha de publicación de la base de datos"),
    ("grado", "nominal", "Grado evaluado (4b)"),
    ("agno", "discreta", "Año del proceso de evaluación (2025)")
]

diccionario = pd.DataFrame(DICCIONARIO_DATA, columns=["variable", "rol_analitico", "descripcion"])
assert len(diccionario) == FICHA["columnas"], "El diccionario debe cubrir las 42 variables exactas."
print(f"✓ Diccionario completo creado exitosamente ({len(diccionario)} variables).")
display(diccionario.head(10))

✓ Diccionario completo creado exitosamente (42 variables).


,variable,rol_analitico,descripcion
0,rbd,identificador,Rol Base de Datos del colegio
1,dvrbd,identificador,Dígito verificador del RBD
2,nom_rbd,alta_cardinalidad,Nombre oficial del establecimiento
3,cod_reg_rbd,nominal,Código numérico de la región
4,nom_reg_rbd,nominal,Nombre de la región geográfica
5,cod_pro_rbd,nominal,Código numérico de la provincia
6,nom_pro_rbd,nominal,Nombre de la provincia
7,cod_com_rbd,identificador,Código de la comuna
8,nom_com_rbd,nominal,Nombre de la comuna
9,cod_deprov_rbd,nominal,Código departamento provincial


In [9]:
ARCHIVO_RAW = DIR_RAW / "simce4b2025_rbd_final.csv"

if ARCHIVO_RAW.exists():
    df_verif = pd.read_csv(ARCHIVO_RAW, sep=";", encoding="latin-1")
    print("✓ Archivo de datos localizado:", ARCHIVO_RAW.as_posix())
    print(f"  Dimensiones reales: {df_verif.shape[0]:,} filas x {df_verif.shape[1]} columnas")
    assert df_verif.shape == (FICHA["filas"], FICHA["columnas"]), "Las dimensiones difieren de la ficha."
    print("  [OK] Coincidencia física y dimensional verificada al 100%.")
else:
    print(f"[PENDIENTE] Mueva el archivo 'simce4b2025_rbd_final.csv' a la carpeta: {DIR_RAW.as_posix()}")

✓ Archivo de datos localizado: data/raw/simce4b2025_rbd_final.csv
  Dimensiones reales: 7,143 filas x 42 columnas
  [OK] Coincidencia física y dimensional verificada al 100%.


In [10]:
def git(*args):
    """Ejecuta un comando de git en el sistema."""
    if shutil.which("git") is None:
        return False, "Git no está instalado."
    res = subprocess.run(["git", *args], capture_output=True, text=True)
    return res.returncode == 0, res.stdout.strip()

ok_git, v_git = git("--version")
print("Diagnóstico Git:", v_git if ok_git else "No detectado")

ok_log, log_git = git("log", "--oneline", "-5")
print("\nÚltimos Commits:")
print(log_git if ok_log and log_git else "Sin commits aún en esta rama.")

Diagnóstico Git: git version 2.55.0.windows.5

Últimos Commits:
4b06d0b Corrige Minuta 2
6e127fd Minuta 1 profe y Minuta 2 grupo
dd7562a feat: actualizacion de minuta
444831b feat: actualizacion
3ef7ee1 feat: actualizacion
